## Standard Dataproc: Cluster Management with command line

This guide will walk you through everything you need to know to successfully run the Dataproc deployment models based on **Dataproc on Compute Engine** (also surfaced as **Managed Apache Spark** in the Google Cloud Console). We'll cover prerequisites, a breakdown of each argument, best practices, and the 4 automation scripts in [`scripts/`](./scripts/):

| Script | Notebook Section Automated |
| :--- | :--- |
| [`scripts/01_create_cluster.sh`](./scripts/01_create_cluster.sh) | **Create a cluster**: Enables APIs, provisions VPC/subnet with Private Google Access, creates `gs://${DATAPROC_BUCKET}`, uploads `data/sales_data.csv` to `gs://${DATAPROC_BUCKET}/data/sales_data.csv`, configures IAM, and creates the Dataproc cluster with Component Gateway & Jupyter. |
| [`scripts/02_update_cluster.sh`](./scripts/02_update_cluster.sh) | **Update a cluster**: Scales primary & secondary (Spot/preemptible) workers, demonstrates YARN Graceful Decommissioning, and updates/removes FinOps resource labels. |
| [`scripts/03_autoscaling_policy.sh`](./scripts/03_autoscaling_policy.sh) | **Autoscaling**: Imports [`config/my_autoscaling_policy.yaml`](./config/my_autoscaling_policy.yaml) and attaches it to the running cluster. |
| [`scripts/04_delete_cluster.sh`](./scripts/04_delete_cluster.sh) | **Delete a cluster**: Configures `--max-idle` auto-termination (`configure-lifecycle`) or immediately deletes the cluster. |

### Prerequisites:

* **Google Cloud Project:** You need an active Google Cloud Project.
* **Billing Enabled:** Billing must be enabled for your project.
* **APIs Enabled:** The Dataproc API, Compute Engine API, and Cloud Storage API must be enabled.
* **gcloud CLI Installed and Configured:** You can run these commands locally or in Cloud Shell.

### Create a cluster (`scripts/01_create_cluster.sh`)

> **Automated Script**: Run all steps in this section automatically with:
> ```bash
> bash 02_cluster_setup_and_management/scripts/01_create_cluster.sh
> ```

Below is the comprehensive set of `gcloud` commands to prepare your project, create the Cloud Storage bucket, upload `sales_data.csv`, and provision the Dataproc cluster.


In [ ]:
# Dataproc workshop CLI / PySpark command
gcloud auth login

REGION=europe-west1
gcloud config set dataproc/region $REGION
PROJECT_ID=$(gcloud config get-value project) && \
gcloud config set project $PROJECT_ID
gcloud config set billing/quota_project $PROJECT_ID
gcloud auth application-default set-quota-project $PROJECT_ID


  * **`REGION = europe-west1`**: This line defines a shell variable `REGION` and sets its value to `europe-west1`. This makes the region easily reusable in subsequent commands.
  * **`gcloud config set dataproc/region $REGION`**: This command sets the default region for Dataproc operations in your `gcloud` configuration. This is convenient because you won't have to explicitly include `--region` in every `gcloud dataproc` command.
  * **`PROJECT_ID=$(gcloud config get-value project) && gcloud config set project $PROJECT_ID
gcloud config set billing/quota_project $PROJECT_ID
gcloud auth application-default set-quota-project $PROJECT_ID`**: These commands ensure that your `gcloud` CLI and Application Default Credentials (ADC) quota project are configured to operate within the correct Google Cloud Project.


In [ ]:
# Dataproc workshop CLI / PySpark command
DATAPROC_BUCKET="${PROJECT_ID}-dataproc-temp"

# 1. Create the Cloud Storage bucket for Dataproc staging and workshop data
gcloud storage buckets create "gs://${DATAPROC_BUCKET}" \
  --location=$REGION \
  --uniform-bucket-level-access \
  --project=$PROJECT_ID

# 2. Upload sample retail sales CSV to gs://${DATAPROC_BUCKET}/data/sales_data.csv
gcloud storage cp 02_cluster_setup_and_management/data/sales_data.csv \
  "gs://${DATAPROC_BUCKET}/data/sales_data.csv" \
  --project=$PROJECT_ID

# 3. Grant Storage Admin on the bucket to the Compute Engine default service account
PROJECT_NUMBER=$(gcloud projects describe $PROJECT_ID --format='value(projectNumber)')
gcloud storage buckets add-iam-policy-binding "gs://${DATAPROC_BUCKET}" \
  --member=serviceAccount:$PROJECT_NUMBER-compute@developer.gserviceaccount.com \
  --role=roles/storage.admin \
  --project=$PROJECT_ID


  * **`gcloud storage buckets create gs://${DATAPROC_BUCKET}` & `gcloud storage cp`**: Creates your Cloud Storage bucket and uploads `data/sales_data.csv` to `gs://${DATAPROC_BUCKET}/data/sales_data.csv` so the PySpark notebook cell (`gs://<BUCKET_NAME>/data/*.csv`) has data ready to read.
  * **`PROJECT_NUMBER=$(gcloud projects describe $PROJECT_ID --format='value(projectNumber)')`**: Retrieves the numerical project number for your Google Cloud Project. This number is part of the default Compute Engine service account name.
  * **`gcloud storage buckets add-iam-policy-binding ... --role=roles/storage.admin`**: Grants the `Storage Admin` role on `gs://${DATAPROC_BUCKET}` to the default Compute Engine service account (`<PROJECT_NUMBER>-compute@developer.gserviceaccount.com`).
      * **Why is this important?** Dataproc clusters validate that this service account has `storage.buckets.get` and `storage.objects.*` permissions on `--bucket` and `--temp-bucket` during cluster creation.


In [ ]:
# Dataproc workshop CLI / PySpark command
gcloud compute networks subnets update default --region=$REGION --enable-private-ip-google-access


  * **`gcloud compute networks subnets update default --region=$REGION --enable-private-ip-google-access`**: This command enables **Private Google Access** for the subnetwork in your specified region (`europe-west1`).
      * **Why is this important, especially with `--no-address`?** When you use the `--no-address` flag with `gcloud dataproc clusters create`, it means your cluster VMs will **not** have external public IP addresses. This enhances security.
      * However, even without public IPs, your cluster VMs often need to communicate with Google Cloud APIs (like Dataproc control plane, Cloud Storage, Cloud Logging, etc.). Private Google Access allows VMs without external IP addresses to send traffic to Google API and service IP addresses through Google's internal network. Without it, `--no-address` clusters might fail to communicate with necessary Google services.
      * While enabling the feature itself is beneficial, using the default subnet for production workloads might be a point of architectural review for very mature or large-scale environments. In such scenarios, best practices often suggest Custom Subnets (`scripts/01_create_cluster.sh` automatically provisions `composer-workshop-vpc` and `composer-workshop-subnet` if needed).


In [ ]:
# Automated script equivalent:
#   bash 02_cluster_setup_and_management/scripts/01_create_cluster.sh

gcloud dataproc clusters create example-dataproc-cluster \
  --enable-component-gateway \
  --bucket=$DATAPROC_BUCKET \
  --temp-bucket=$DATAPROC_BUCKET \
  --region=europe-west1 \
  --no-address \
  --master-machine-type=e2-standard-4 \
  --num-workers=2 \
  --worker-machine-type=e2-standard-4 \
  --worker-boot-disk-size=500 \
  --image-version=2.2-debian12 \
  --optional-components=JUPYTER \
  --max-age=3600s \
  --labels=mode=workshop,user=eilinlux \
  --project=$PROJECT_ID


This is the core command to create your Dataproc cluster. Let's look at the key flags:

  * **`--bucket=$DATAPROC_BUCKET` & `--temp-bucket=$DATAPROC_BUCKET`**: Explicitly points both the Dataproc staging bucket and temporary scratch bucket to the Cloud Storage bucket where we granted `roles/storage.admin`.
  * **`--enable-component-gateway`**: Explicitly enables the Component Gateway. The Component Gateway acts as a secure proxy to the web interfaces running on your cluster's master node (like JupyterLab, YARN ResourceManager UI, and Spark History Server).
  * **`--image-version 2.2-debian12`**: Specifies a Dataproc image based on Debian 12 with Dataproc version 2.2. Check the available list at: https://cloud.google.com/dataproc/docs/concepts/versioning/dataproc-version-clusters
  * **`--no-address`**: Creates your cluster VMs without external public IP addresses, enhancing security. Requires Private Google Access to be enabled on the subnetwork.
  * **`--optional-components=JUPYTER`**: Installs Jupyter and JupyterLab on the cluster's master node.
  * The other flags (**`--worker-boot-disk-size`**, **`--worker-machine-type`**, **`--master-machine-type`**) define the size and type of the virtual machines for your cluster.


### UseCase 1: Jupyter Notebooks on Dataproc

You've taken all the right steps for a secure and functional Dataproc cluster with Jupyter:

1.  **Region and Project Setup:** Confirmed.
2.  **Cloud Storage Bucket & CSV Upload:** Created `gs://${DATAPROC_BUCKET}`, uploaded `data/sales_data.csv` to `gs://${DATAPROC_BUCKET}/data/sales_data.csv`, and granted `Storage Admin` permissions to the Compute Engine service account.
3.  **Private Google Access:** Enabled, which is essential for `--no-address` clusters to communicate with Google services.
4.  **Cluster Creation:**
      * `example-dataproc-cluster` created.
      * `--enable-component-gateway` for UI access.
      * Specific `2.2-debian12` image version used.
      * `--no-address` for enhanced security.
      * `JUPYTER` component included.

**How to open Jupyter Notebook in the UI:**

1.  **Go to the Google Cloud Console:** `https://console.cloud.google.com`
2.  **Navigate to Dataproc Clusters:** From the navigation menu (☰), go to **Analytics \> Dataproc (or Managed Apache Spark) \> Clusters**.
3.  **Select Your Cluster:** Click on the name of your cluster, `example-dataproc-cluster`.
4.  **Go to "Web Interfaces" Tab:** On the cluster details page, click the **Web Interfaces** tab.
5.  **Click Jupyter/JupyterLab Link:** You will see a link for **Jupyter** or **JupyterLab**. Click on it to open the Jupyter interface in a new browser tab.


![image.png](../artifacts/04_dataproc_serverless_batches_ui.png)


6.  **Create a New Notebook or Run Locally:**
    * In the Dataproc JupyterLab interface, select **New** -> **PySpark** (or Python 3).
    * Alternatively, run the cell below directly in your local IDE using the **`Python (dataproc-workshop)`** kernel (`uv sync`).

7.  **Run Your PySpark Code:**
    * Running `bash 02_cluster_setup_and_management/scripts/01_create_cluster.sh` automatically uploads `data/sales_data.csv` to `gs://${DATAPROC_BUCKET}/data/sales_data.csv`.
    * When executed locally in your IDE (where the Dataproc GCS connector JAR is not bundled), the cell below automatically reads the local [`data/sales_data.csv`](./data/sales_data.csv) file; when executed on the Dataproc cluster, it reads `gs://${DATAPROC_BUCKET}/data/*.csv`.


In [ ]:
# Dataproc workshop CLI / PySpark command
import os
from pathlib import Path
from dotenv import load_dotenv
from pyspark.sql import SparkSession

# Load DATAPROC_BUCKET from the repository root .env if present
for env_candidate in [Path.cwd() / ".env", Path.cwd().parent / ".env"]:
    if env_candidate.exists():
        load_dotenv(env_candidate)
        break

# Get or create a SparkSession
spark = SparkSession.builder.appName("MyDataprocNotebook").getOrCreate()

# Cloud Storage path populated by `bash 02_cluster_setup_and_management/scripts/01_create_cluster.sh`
bucket_name = os.environ.get("DATAPROC_BUCKET", "<BUCKET_NAME>")
gcs_path = f"gs://{bucket_name}/data/*.csv"

# Use local data/sales_data.csv when running in a local IDE checkout, or gs:// on Dataproc
local_candidates = [Path("data/sales_data.csv"), Path("02_cluster_setup_and_management/data/sales_data.csv")]
local_csv = next((p for p in local_candidates if p.exists()), None)
csv_path = str(local_csv) if local_csv is not None else gcs_path

print(f"Reading retail sales CSV from: {csv_path} (GCS target: {gcs_path})")
df = spark.read.csv(csv_path, header=True, inferSchema=True)

df.show()


### Update a cluster (`scripts/02_update_cluster.sh`)

> **Automated Script**: Run all live cluster update operations below (scaling primary & secondary workers, graceful decommissioning, and updating/removing FinOps labels) with:
> ```bash
> bash 02_cluster_setup_and_management/scripts/02_update_cluster.sh
> ```

You can update a running Google Cloud Dataproc cluster to modify its resources, primarily for scaling or managing metadata. The most common tool for this is the `gcloud` command-line interface.

It is important to understand that not all cluster properties can be updated. Core configurations like the machine type, image version, and network settings cannot be changed on a running cluster. For those changes, you must create a new cluster with the desired configuration.

Here are the most common update operations you can perform on a live Dataproc cluster:

#### **Scaling the Number of Worker Nodes:** 
This is the most frequent update operation, allowing you to add or remove worker nodes to adjust to your workload's demands (scaling out or in).


In [ ]:
# Dataproc workshop CLI / PySpark command
gcloud dataproc clusters update example-dataproc-cluster \
    --region=europe-west1 \
    --num-workers=3 \
    --num-secondary-workers=2


#### **Set a specific number of secondary (preemptible) workers:** 

To set a specific number of secondary (preemptible/Spot) workers, you use the `--num-secondary-workers` flag when creating or updating your Dataproc cluster. These workers are cost-effective but can be preempted (shut down) by Google Cloud, making them suitable for fault-tolerant workloads.


In [ ]:
# Dataproc workshop CLI / PySpark command
gcloud dataproc clusters create example-cluster \
    --region=europe-west1 \
    --zone=europe-west1-b \
    --master-machine-type=e2-standard-4 \
    --worker-machine-type=e2-standard-4 \
    --num-workers=2 \
    --num-secondary-workers=4 \
    --secondary-worker-machine-type=e2-standard-4 \
    --image-version=2.2-debian12 \
    --scopes=cloud-platform \
    --project=$PROJECT_ID


**Explanation of New/Modified Parameters:**

  * **`--num-workers=2`**: This remains the number of **primary (on-demand) workers**. These workers are stable and not subject to preemption.
  * **`--num-secondary-workers=4`**: This is the new flag. It specifies that you want **4 secondary (preemptible) workers** in addition to your primary workers.
  * **`--secondary-worker-machine-type=e2-standard-4`**: (Optional, but good practice to specify if different from primary) This flag allows you to define the machine type for your secondary workers. In this example, it's the same as the primary workers, but you could use a different type if desired (e.g., smaller instances for burst capacity).

**Why use secondary (preemptible) workers?**

  * **Cost Savings:** Preemptible VMs are significantly cheaper (up to 80% off) than standard VMs.
  * **Scalability for Bursty Workloads:** They allow you to add a large amount of compute power for a fraction of the cost, ideal for jobs that can tolerate worker loss and retries (which Spark is generally good at).
  * **Improved Throughput:** More workers mean more parallel processing, potentially reducing job completion times for large datasets.

**Considerations when using secondary workers:**

  * **Preemption:** Secondary workers can be shut down by Google Cloud with a 30-second warning. Spark jobs should be designed to handle this gracefully (e.g., by recomputing lost partitions, which Spark's fault tolerance generally does automatically).
  * **Stateful Operations:** Avoid using secondary workers for highly stateful operations that would be severely impacted by worker loss.
  * **Driver Node:** Your master node (driver) should always be a primary (on-demand) VM.


#### **Using Graceful Decommissioning**
When scaling down, you can use graceful decommissioning to ensure that running tasks on the nodes being removed are not lost. This feature tells YARN to stop scheduling new tasks on those nodes and allows existing tasks to finish within a specified timeout.

In this example, Dataproc will wait up to 3600 seconds (1 hour) for the work on the nodes being removed to complete before shutting them down.


In [ ]:
# Dataproc workshop CLI / PySpark command
gcloud dataproc clusters update example-dataproc-cluster \
    --region=europe-west1 \
    --num-workers=2 \
    --num-secondary-workers=0 \
    --graceful-decommission-timeout=3600s


#### **Updating Labels**
Labels are key-value pairs that help you organize and manage your GCP resources, and assign costs based on a FinOps approach. You can add, update, or remove labels on a running cluster.

To add or update labels, this command will add these labels or update their values if the keys already exist.


In [ ]:
# Dataproc workshop CLI / PySpark command
gcloud dataproc clusters update example-dataproc-cluster \
    --region=europe-west1 \
    --update-labels=env=production,cost_center=research,user=eilinlux



To remove a label:


In [ ]:
# Dataproc workshop CLI / PySpark command
gcloud dataproc clusters update example-dataproc-cluster \
    --region=europe-west1 \
    --remove-labels=cost_center


## Autoscaling (`scripts/03_autoscaling_policy.sh`)

> **Automated Script & YAML Config**: The YAML policy below is stored at [`config/my_autoscaling_policy.yaml`](./config/my_autoscaling_policy.yaml), and you can import and attach it to your running cluster with:
> ```bash
> bash 02_cluster_setup_and_management/scripts/03_autoscaling_policy.sh
> ```

If you have an autoscaling policy attached to your cluster, you can update it to modify its scaling behavior. First, you might update the policy itself, and then you can re-apply it or attach a new one.

This policy is a standalone resource in GCP that defines the logic for automatically adding or removing worker nodes from your cluster. The policy contains rules like:
1. **Scaling Limits:**
- The minimum number of workers the cluster must always have.
- The maximum number of workers the cluster is allowed to scale up to (to control costs).
2. **Scale-Up Rules (Adding Workers):** The trigger for adding more nodes. This is typically based on YARN metrics, like how much memory is pending for allocation. For example: "If the cluster needs more memory than is available for 1 minute, start adding workers."
3. **Scale-Down Rules (Removing Workers):** The trigger for removing idle nodes. For example: "If there are idle nodes with no running tasks for 10 minutes, start removing them."
4. **Cooldown Period:** A waiting period after a scaling action to prevent the cluster from scaling up and down too rapidly (known as "flapping").

The key benefit is that you can create one policy and apply it to multiple clusters, ensuring consistent scaling behavior everywhere.

### Define the Autoscaling Policy in a YAML File

See [`config/my_autoscaling_policy.yaml`](./config/my_autoscaling_policy.yaml) with the following content:


In [ ]:
# config/my_autoscaling_policy.yaml
id: my-workshop-autoscaling-policy-yaml # A unique ID for the policy
basicAlgorithm:
  cooldownPeriod: 120s # 2 minutes
  yarnConfig:
    gracefulDecommissionTimeout: 60s # 1 minute
    scaleUpFactor: 1.0 # Aggressive scale-up
    scaleDownFactor: 0.5 # Conservative scale-down
workerConfig:
  minInstances: 2
  maxInstances: 10
secondaryWorkerConfig: # Explicitly define secondary worker config, allowing 0 min
  minInstances: 0
  maxInstances: 10 # Set a reasonable max for secondary workers if you might use them


**Explanation of YAML Fields:**

  * **`id`**: This is the unique identifier for your autoscaling policy. It will be used to refer to this policy when creating or updating a cluster.
  * **`basicAlgorithm`**: Defines the core logic for autoscaling.
      * **`cooldownPeriod`**: The time (in seconds, with `s` suffix) to wait between scaling operations. Prevents rapid scaling up and down.
      * **`yarnConfig`**: Specific configurations for YARN-based autoscaling.
          * **`gracefulDecommissionTimeout`**: Time (in seconds) to wait for YARN applications to finish on a node before it's removed during scale-down.
          * **`scaleUpFactor`**: A float (0.0 to 1.0). How aggressively to scale up when pending resources are detected. 1.0 means scale up to eliminate all pending resources.
          * **`scaleDownFactor`**: A float (0.0 to 1.0). How aggressively to scale down when excess resources are detected. 0.5 means remove half of the excess.
  * **`workerConfig`**: Defines the scaling parameters for **primary workers**.
      * **`minInstances`**: Minimum number of primary worker nodes.
      * **`maxInstances`**: Maximum number of primary worker nodes.
  * **`secondaryWorkerConfig`**: Defines the scaling parameters for **secondary (preemptible/spot) workers**.
      * **`minInstances`**: Minimum number of secondary worker nodes. Setting this to `0` allows you to create clusters without secondary workers.
      * **`maxInstances`**: Maximum number of secondary worker nodes.

### Create the Autoscaling Policy from the YAML File

Once you have your `config/my_autoscaling_policy.yaml` file, you can create the policy using the `gcloud dataproc autoscaling-policies import` command:


In [ ]:
# Dataproc workshop CLI / PySpark command
gcloud dataproc autoscaling-policies import my-workshop-autoscaling-policy-yaml \
    --source=02_cluster_setup_and_management/config/my_autoscaling_policy.yaml \
    --region=europe-west1


  * `my-workshop-autoscaling-policy-yaml`: This is the `policy_id` that you defined in the YAML file (`id` field). It will be used to refer to this policy.
  * `--source=02_cluster_setup_and_management/config/my_autoscaling_policy.yaml`: Specifies the path to your YAML file.
  * `--region=europe-west1`: The region where the policy should be created.

### Attach the Policy to Your Dataproc Cluster

After the policy is successfully created, you can attach it to your new or existing Dataproc cluster using the `--autoscaling-policy` flag, referencing the `policy_id` you defined in the YAML.

**For a new cluster:**


In [ ]:
# Dataproc workshop CLI / PySpark command
gcloud dataproc clusters create ex-cluster-autoscaled-from-yaml \
    --enable-component-gateway \
    --bucket=$DATAPROC_BUCKET \
    --temp-bucket=$DATAPROC_BUCKET \
    --region=europe-west1 \
    --no-address \
    --master-machine-type=e2-standard-4 \
    --num-workers=2 \
    --worker-machine-type=e2-standard-4 \
    --worker-boot-disk-size=500 \
    --image-version=2.2-debian12 \
    --optional-components=JUPYTER \
    --max-age=3600s \
    --labels=mode=workshop,user=eilinlux \
    --autoscaling-policy=my-workshop-autoscaling-policy-yaml \
    --project=$PROJECT_ID


**To update an existing cluster (automated in `scripts/03_autoscaling_policy.sh`):**

If you already have a running cluster (`example-dataproc-cluster`) and want to attach this policy to it:


In [ ]:
# Dataproc workshop CLI / PySpark command
gcloud dataproc clusters update example-dataproc-cluster \
    --region=europe-west1 \
    --autoscaling-policy=my-workshop-autoscaling-policy-yaml


## Delete a cluster (`scripts/04_delete_cluster.sh`)

> **Automated Script**: You can configure `--max-idle` auto-termination or immediately delete the cluster using [`scripts/04_delete_cluster.sh`](./scripts/04_delete_cluster.sh):
> ```bash
> # Option A: Configure 30-minute idle auto-deletion (--max-idle=30m) on the live cluster
> bash 02_cluster_setup_and_management/scripts/04_delete_cluster.sh configure-lifecycle
>
> # Option B: Immediately delete the cluster when finished
> bash 02_cluster_setup_and_management/scripts/04_delete_cluster.sh
> ```

A **Google Dataproc cluster does not delete itself by default.** Once you create a standard cluster, it will continue to run and incur costs until you manually delete it.

To have a cluster delete itself automatically, you can configure lifecycle deletion at creation time or via `gcloud dataproc clusters update`. There are two primary ways to do this:

#### **Scheduled Deletion (Based on Age)**
This feature automatically deletes the cluster after a specific amount of time has passed since its creation. It's like setting a self-destruct timer.

- **How it works:** You specify a maximum age for the cluster. Once the cluster reaches this age, Dataproc will automatically start the deletion process, regardless of whether it is running a job or is idle.
- **Use Case:** This is ideal for jobs with a predictable maximum runtime. For example, if you know your data processing task will never take more than 4 hours, you can set the max age to 4h to ensure you don't forget to delete it.
- **Command Flag:** `--max-age`
- **Example:** This command creates a cluster that will automatically delete itself after 4 hours, or deletes a cluster immediately:


In [ ]:
# Dataproc workshop CLI / PySpark command
gcloud dataproc clusters create my-temp-cluster \
    --region=europe-west1 \
    --max-age=4h \
    --num-workers=2

gcloud dataproc clusters delete example-dataproc-cluster --region=europe-west1


#### **Deletion Based on Idle Time**
This feature automatically deletes the cluster after it has been "idle" (not running any jobs) for a specified period.
- **How it works:** Dataproc monitors the cluster for active jobs. If no jobs are running for the duration you set, the cluster is considered idle and the deletion process begins. If a new job is submitted, the idle timer resets.
- **Use Case:** This is perfect for ad-hoc or interactive clusters, like a cluster used for development or data exploration with a Jupyter notebook. You use it when you need it, and if you forget to shut it down, it will clean itself up after a period of inactivity.
- **Command Flag:** `--max-idle`
- **Example:** This command updates a running cluster (or creates a new cluster) so that it automatically deletes itself after being idle for 30 minutes:


In [ ]:
# Dataproc workshop CLI / PySpark command
gcloud dataproc clusters update example-dataproc-cluster \
    --region=europe-west1 \
    --max-idle=30m


**Summary of Differences**

| Feature | `--max-age` (Scheduled Deletion) | `--max-idle` (Max Idle Time) |
|---|---|----|
| Trigger | Time since creation (cluster's age) | Time since the last job finished (idle duration) |
| Ideal For | Batch jobs with predictable runtimes | Interactive, ad-hoc, or development clusters |
| Behavior | Deletes even if a job is still running | Only deletes when the cluster is not running jobs |

A third, more advanced option is to use Dataproc Workflow Templates (or **Dataproc Serverless / Google Cloud Serverless for Apache Spark** in Chapter 04). In a Workflow Template, you define a series of jobs as a workflow, and the template automatically creates an ephemeral cluster, runs your jobs, and deletes the cluster upon completion.
